In [ ]:
from pathlib import Path
import time

from common.setup_plotting import setup_matplotlib, get_figure_dir
from common.training import save_checkpoint, load_checkpoint

import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
from tqdm.notebook import tqdm

from denoising_diffusion_pytorch import Unet, GaussianDiffusion

setup_matplotlib()        # configure matplotlib first (So i can use LaTeX in the labels)
# Make interactive plots work in Jupyter notebooks
%matplotlib inline
import matplotlib.pyplot as plt   # THEN import pyplot

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

fig_dir = get_figure_dir("task_07")
data_dir = Path("../../../data/task_07")
generated_dir = data_dir / "generated_samples"
checkpoint_path = Path("../models/mnist_ddpm_library.pt")

data_dir.mkdir(parents=True, exist_ok=True)
generated_dir.mkdir(parents=True, exist_ok=True)

print("device:", device)
print("figure directory:", fig_dir)
print("generated samples directory:", generated_dir)

# Task 07: DDPM on MNIST

This notebook trains a denoising diffusion probabilistic model (DDPM) on MNIST. During training, clean digit images $x_0$ are corrupted with Gaussian noise at random timesteps $t$, and a U-Net learns to predict the noise that was added. During sampling, the model starts from random noise and repeatedly applies the learned denoising steps until an image-like sample remains.

For this task the MNIST tensors are kept in the pixel range $[0, 1]$. The diffusion wrapper is configured explicitly so it does not apply an additional automatic normalization step.

In [ ]:
learning_rate = 4e-4
batch_size = 128
num_epochs = 50
max_batches_per_epoch = None

image_size = 28
image_channels = 1
num_sample_images = 64
sample_every = 10

time_steps = 1000
sampling_timesteps = 250
beta_schedule = "linear"
objective = "pred_noise"

unet_dim = 32
unet_dim_mults = (1, 2, 4)
use_flash_attention = False

skip_training = True

print("batch_size:", batch_size)
print("num_epochs:", num_epochs)
print("max_batches_per_epoch:", max_batches_per_epoch)
print("diffusion steps:", time_steps)
print("sampling steps:", sampling_timesteps)
print("objective:", objective)

In [ ]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(
    root=data_dir,
    train=True,
    transform=mnist_transform,
    download=True,
)

generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=generator,
)

real_images, real_labels = next(iter(train_loader))

print("number of training images:", len(train_dataset))
print("number of batches per epoch:", len(train_loader))
print("image batch shape:", real_images.shape)
print("label batch shape:", real_labels.shape)
print("pixel range:", float(real_images.min()), "to", float(real_images.max()))

In [ ]:
real_grid = utils.make_grid(
    real_images[:num_sample_images],
    nrow=8,
    padding=2,
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(real_grid.permute(1, 2, 0), cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title("Real MNIST Samples")
ax.axis("off")

fig.tight_layout()
fig.savefig(fig_dir / "real_mnist_samples.pdf")
None

In [ ]:
model = Unet(
    dim=unet_dim,
    dim_mults=unet_dim_mults,
    channels=image_channels,
    flash_attn=use_flash_attention,
).to(device)

diffusion = GaussianDiffusion(
    model,
    image_size=image_size,
    timesteps=time_steps,
    sampling_timesteps=sampling_timesteps,
    objective=objective,
    beta_schedule=beta_schedule,
    auto_normalize=False,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model.__class__.__name__)
print("trainable parameters:", num_parameters)
print("image size:", diffusion.image_size)
print("channels:", diffusion.channels)
print("auto_normalize:", False)
print("using DDIM sampling:", diffusion.is_ddim_sampling)

In [ ]:
forward_timesteps = [0, 100, 300, 600, 999]
images_to_noise = real_images[:8].to(device)

noisy_rows = []

with torch.no_grad():
    for step in forward_timesteps:
        timesteps = torch.full(
            (images_to_noise.shape[0],),
            step,
            device=device,
            dtype=torch.long,
        )
        noisy_images = diffusion.q_sample(images_to_noise, timesteps)
        noisy_rows.append(torch.clamp(noisy_images.cpu(), 0.0, 1.0))

forward_grid = utils.make_grid(
    torch.cat(noisy_rows, dim=0),
    nrow=images_to_noise.shape[0],
    padding=2,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(forward_grid.permute(1, 2, 0), cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title("Forward Diffusion on MNIST")
ax.axis("off")

fig.tight_layout()
fig.savefig(fig_dir / "forward_diffusion_mnist.pdf")
None

In [ ]:
def show_image_grid(images, title, filename_stem=None):
    images = torch.clamp(images.detach().cpu(), 0.0, 1.0)

    image_grid = utils.make_grid(
        images[:num_sample_images],
        nrow=8,
        padding=2,
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image_grid.permute(1, 2, 0), cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(title)
    ax.axis("off")
    fig.tight_layout()

    if filename_stem is not None:
        fig.savefig(fig_dir / f"{filename_stem}.pdf")
        utils.save_image(image_grid, generated_dir / f"{filename_stem}.png")

    return fig, ax


@torch.no_grad()
def sample_and_show(title, filename_stem=None):
    diffusion.eval()
    sampled_images = diffusion.sample(batch_size=num_sample_images)
    return show_image_grid(sampled_images, title, filename_stem)

In [ ]:
sample_and_show(
    title="DDPM Samples Before Training",
    filename_stem="ddpm_samples_epoch_00",
)
None

In [ ]:
training_losses = []

if skip_training and checkpoint_path.exists():
    checkpoint = load_checkpoint(
        checkpoint_path,
        model=model,
        optimizer=optimizer,
        device=device,
    )
    training_losses = checkpoint["train_losses"]
    print("Loaded checkpoint from:", checkpoint_path)
    print("Completed epochs:", checkpoint["epoch"])

    sample_and_show(
        title=f"DDPM Samples From Epoch {checkpoint['epoch']}",
        filename_stem=f"ddpm_samples_epoch_{checkpoint['epoch']:02d}_loaded",
    )

else:
    start_time = time.perf_counter()

    for epoch in range(num_epochs):
        diffusion.train()
        epoch_losses = []

        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs}",
            leave=False,
        )

        for batch_idx, (images, _) in enumerate(progress_bar):
            if max_batches_per_epoch is not None and batch_idx >= max_batches_per_epoch:
                break

            images = images.to(device)

            optimizer.zero_grad()
            loss = diffusion(images)
            loss.backward()
            optimizer.step()

            loss_value = loss.item()
            epoch_losses.append(loss_value)
            progress_bar.set_postfix(loss=f"{loss_value:.4f}")

        train_loss = float(np.mean(epoch_losses))
        training_losses.append(train_loss)

        elapsed_minutes = (time.perf_counter() - start_time) / 60
        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"loss: {train_loss:.4f} | "
            f"elapsed: {elapsed_minutes:.1f} min"
        )

        save_checkpoint(
            path=checkpoint_path,
            model=model,
            optimizer=optimizer,
            epoch=epoch + 1,
            train_losses=training_losses,
            val_losses=[],
            best_val_loss=train_loss,
            extra={
                "hyperparameters": {
                    "learning_rate": learning_rate,
                    "batch_size": batch_size,
                    "num_epochs": num_epochs,
                    "max_batches_per_epoch": max_batches_per_epoch,
                    "time_steps": time_steps,
                    "sampling_timesteps": sampling_timesteps,
                    "beta_schedule": beta_schedule,
                    "objective": objective,
                    "unet_dim": unet_dim,
                    "unet_dim_mults": unet_dim_mults,
                }
            },
        )

        if (epoch + 1) % sample_every == 0 or (epoch + 1) == num_epochs:
            sample_and_show(
                title=f"DDPM Samples After Epoch {epoch + 1}",
                filename_stem=f"ddpm_samples_epoch_{epoch + 1:02d}",
            )
            plt.show()

print("training losses:", training_losses)

In [ ]:
first_epoch_loss = training_losses[0]
plot_epochs = np.arange(2, len(training_losses) + 1)
plot_losses = training_losses[1:]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(plot_epochs, plot_losses, marker="o", color="C0", label="training loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Noise prediction loss")
ax.set_title("DDPM Training Loss After First Epoch")
ax.legend()

fig.tight_layout()
fig.savefig(fig_dir / "ddpm_training_loss.pdf")

print("excluded first epoch loss:", first_epoch_loss)
print("final training loss:", training_losses[-1])
None

In [ ]:
sample_and_show(
    title="DDPM Final Generated Samples",
    filename_stem="ddpm_final_generated_samples",
)
None

In [ ]:
torch.manual_seed(123)

with torch.no_grad():
    denoising_history = diffusion.sample(
        batch_size=8,
        return_all_timesteps=True,
    )

snapshot_indices = torch.linspace(
    0,
    denoising_history.shape[1] - 1,
    steps=6,
).long()

denoising_snapshots = denoising_history[:, snapshot_indices]
denoising_snapshots = denoising_snapshots.permute(1, 0, 2, 3, 4)
denoising_snapshots = denoising_snapshots.reshape(-1, image_channels, image_size, image_size)

denoising_grid = utils.make_grid(
    torch.clamp(denoising_snapshots.cpu(), 0.0, 1.0),
    nrow=8,
    padding=2,
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(denoising_grid.permute(1, 2, 0), cmap="gray", vmin=0.0, vmax=1.0)
ax.set_title("Reverse Diffusion Snapshots")
ax.axis("off")

fig.tight_layout()
fig.savefig(fig_dir / "reverse_diffusion_snapshots.pdf")
utils.save_image(denoising_grid, generated_dir / "reverse_diffusion_snapshots.png")
None

### Conclusion

This notebook implements a DDPM for MNIST using a U-Net from `denoising-diffusion-pytorch`. The model is trained as a noise predictor: clean MNIST images are noised at random diffusion timesteps, and the U-Net learns to predict the Gaussian noise component. Sampling starts from random noise and applies the learned reverse process to produce digit-like images.

The first run uses a deliberately conservative runtime setting with `max_batches_per_epoch = 200`. This is enough to check that the training loop, checkpointing, sample generation, and report plots work correctly. Better samples should be possible by increasing the number of epochs, using more batches per epoch, training a larger U-Net, or experimenting with the noise schedule and sampling steps.

The generated images should be interpreted qualitatively. A decreasing noise-prediction loss indicates that the model is learning the denoising task, but final image quality still depends strongly on training time and model capacity.